In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

import src.utils.viz_utils as vu

os.chdir("..")
os.getcwd()

In [ ]:
DATASET = "satbird-USA-summer"  # 's2bms' or 'satbird-USA-summer'
DATA_DIR = os.environ.get("DATA_DIR", "data/")

if DATASET == "s2bms":
    aef_size = 256
    unlabelled = True

    # aux_df = pd.read_csv(f'{DATA_DIR}/s2bms/model_ready_s2bms{"-unlabelled-merged" if unlabelled else ""}.csv')
    aux_df = pd.read_csv(
        f'{DATA_DIR}/s2bms/model_ready_s2bms{"-unlabelled-merged_incl-test-aux" if unlabelled else ""}.csv'
    )
    aef_df = pd.read_csv(
        f'{DATA_DIR}/s2bms/eo/avr_aef_{aef_size}{"_unlabelled" if unlabelled else ""}.csv'
    )

    import torch  # only used here, to read the .pth split-index file; no tensors kept downstream

    # split_indices = torch.load(
    #     os.path.join(f'{DATA_DIR}/s2bms/splits/s2bms{"_unlabelled" if unlabelled else ""}_union_val_test.pth'),
    #     weights_only=False,
    # )
    split_indices = torch.load(
        os.path.join(
            f"{DATA_DIR}/s2bms/splits/split_indices_s2bms+s2bms-unlabelled-20260529_2026-05-29-1438.pth"
        ),
        weights_only=False,
    )
    aux_idx = aux_df.name_loc
    aef_idx = aef_df.name_loc

    common_train_idx = pd.Series(
        list(set(split_indices["train_indices"]) & set(aef_idx) & set(aux_idx))
    )
    common_val_idx = pd.Series(
        list(set(split_indices["val_indices"]) & set(aef_idx) & set(aux_idx))
    )
    common_test_idx = pd.Series(
        list(set(split_indices["test_indices"]) & set(aef_idx) & set(aux_idx))
    )

elif DATASET == "satbird-USA-summer":
    aef_size = 128

    aef_df = pd.read_csv(
        f"{DATA_DIR}/satbird-USA-summer/eo/aef-{DATASET.lower()}_average-{aef_size}.csv"
    )
    aux_df = pd.read_csv(
        f"{DATA_DIR}/satbird-USA-summer/model_ready_satbird-USA-summer_with_lc.csv"
    )

    common_names = set(aef_df.name_loc) & set(aux_df.name_loc)
    aef_df = (
        aef_df[aef_df.name_loc.isin(common_names)].sort_values("name_loc").reset_index(drop=True)
    )
    aux_df = (
        aux_df[aux_df.name_loc.isin(common_names)].sort_values("name_loc").reset_index(drop=True)
    )

    ## remove rows with nans
    ind_row_nans = aux_df[aux_df.isna().any(axis=1)].index
    if len(ind_row_nans) > 0:
        print(f"Removing {len(ind_row_nans)} rows with NaNs in aux_df")
        aux_df = aux_df.drop(ind_row_nans)
        aef_df = aef_df.drop(ind_row_nans)

    aux_df = aux_df.reset_index(drop=True)
    aef_df = aef_df.reset_index(drop=True)

    assert all(
        aux_df.name_loc.values == aef_df.name_loc.values
    ), "name_loc mismatch between aux_df and aef_df"

    common_train_idx = aux_df[aux_df.split == "train"].name_loc
    common_val_idx = aux_df[aux_df.split == "valid"].name_loc
    common_test_idx = aux_df[aux_df.split == "test"].name_loc

print(
    "train | validation | test\n",
    len(common_train_idx),
    "|",
    len(common_val_idx),
    "|",
    len(common_test_idx),
)

In [ ]:
def get_split(name_loc_set):
    aef_sub = aef_df[aef_df["name_loc"].isin(map(str, name_loc_set))].sort_values("name_loc")
    aux_sub = aux_df[aux_df["name_loc"].isin(map(str, name_loc_set))].sort_values("name_loc")
    return aef_sub, aux_sub


emb_cols = [f"emb_{i}" for i in range(64)]  # must match aef_size
# aux_cols = [c for c in aux_df.columns if c.startswith("aux_") and "top" not in c]
aux_cols = [c for c in aux_df.columns if "aux" in c and "top" not in c]

aef_train_df, aux_train_df = get_split(common_train_idx)
aef_val_df, aux_val_df = get_split(common_val_idx)
aef_test_df, aux_test_df = get_split(common_test_idx)

aef_train = aef_train_df[emb_cols].to_numpy()
aef_val = aef_val_df[emb_cols].to_numpy()
aef_test = aef_test_df[emb_cols].to_numpy()

aux_train = aux_train_df[aux_cols].to_numpy()
aux_val = aux_val_df[aux_cols].to_numpy()
aux_test = aux_test_df[aux_cols].to_numpy()

print(aef_train.shape, aef_val.shape, aef_test.shape)
print(aux_train.shape, aux_val.shape, aux_test.shape)

In [ ]:
# Compute on RAW (pre-filter, 87-column) arrays -- before any keep_mask slicing
aux_std_raw = aux_train.std(axis=0)
train_nonzero_frac = (aux_train != 0).mean(axis=0)
val_nonzero_frac = (aux_val != 0).mean(axis=0)
test_nonzero_frac = (aux_test != 0).mean(axis=0)
min_std = 1e-2  # because of different LC
min_nonzero_frac = 1e-2

keep_mask = (
    (aux_std_raw > min_std)
    & (train_nonzero_frac > min_nonzero_frac)
    & (val_nonzero_frac > min_nonzero_frac)
    & (test_nonzero_frac > min_nonzero_frac)
)
dropped_names = [name for name, keep in zip(aux_cols, keep_mask) if not keep]
print(
    f"Dropping {len(dropped_names)}/{len(aux_cols)} low-variance/sparse aux columns: {dropped_names}"
)

kept_names = [name for name, keep in zip(aux_cols, keep_mask) if keep]

coverage_df = pd.DataFrame(
    {
        "AUX_Variable": kept_names,
        "train_std": aux_std_raw[keep_mask],
        "pct_nonzero_train": train_nonzero_frac[keep_mask] * 100,
        "pct_nonzero_val": val_nonzero_frac[keep_mask] * 100,
        "pct_nonzero_test": test_nonzero_frac[keep_mask] * 100,
    }
).sort_values("pct_nonzero_val")
# print(coverage_df.head(15))

# NOW subset, once, after keep_mask is finalized
aux_train, aux_val, aux_test = (
    aux_train[:, keep_mask],
    aux_val[:, keep_mask],
    aux_test[:, keep_mask],
)

# kept_names

## Linear probing 

In [ ]:
# Standardize using TRAIN stats only
aef_scaler = StandardScaler().fit(aef_train)
X_train = aef_scaler.transform(aef_train)
X_val = aef_scaler.transform(aef_val)
X_test = aef_scaler.transform(aef_test)

aux_scaler = StandardScaler().fit(aux_train)
y_train = aux_scaler.transform(aux_train)
y_val = aux_scaler.transform(aux_val)
y_test = aux_scaler.transform(aux_test)

print("Shape of X_train and y_train:", X_train.shape, y_train.shape)

In [ ]:
# Ridge regression
# alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
alphas = [1]

r2_scores = []
best_alphas = []

for i in range(y_train.shape[1]):
    y_target_train = y_train[:, i]
    y_target_val = y_val[:, i]
    y_target_test = y_test[:, i]

    # pick alpha using the spatial val split, not sklearn's internal random CV
    val_r2_per_alpha = []
    for a in alphas:
        m = Ridge(alpha=a)
        m.fit(X_train, y_target_train)
        val_r2_per_alpha.append(r2_score(y_target_val, m.predict(X_val)))

    best_alpha = alphas[int(np.argmax(val_r2_per_alpha))]
    best_alphas.append(best_alpha)

    final_model = Ridge(alpha=best_alpha)
    final_model.fit(X_train, y_target_train)
    y_pred = final_model.predict(X_test)

    r2_scores.append(r2_score(y_target_test, y_pred))


results_df = pd.DataFrame(
    {
        "AUX_Variable": kept_names,
        "Test_R2": r2_scores,
        "Best_Alpha": best_alphas,
    }
)

plot_var = "Test_R2"
df_sorted = results_df.sort_values(plot_var, ascending=False)
# print(df_sorted)
# df_sorted.to_csv(f'../ridge_reg_avr_aef_{aef_size}_to_aux_unlabelled.csv', index=False)

df_sorted["lc_sum"] = np.nan
df_sorted["lc_n_above_threshold"] = np.nan

dict_sum_lc = {name: sum_val for name, sum_val in zip(kept_names, aux_train.sum(0))}

threshold_cover = 0.2

for k, v in dict_sum_lc.items():
    if "corine" in k:
        df_sorted.loc[df_sorted["AUX_Variable"] == k, "lc_sum"] = v
        ## get number of locations with k values above threshold_cover
        df_sorted.loc[df_sorted["AUX_Variable"] == k, "lc_n_above_threshold"] = (
            aux_train[:, kept_names.index(k)] > threshold_cover
        ).sum()


fig, ax = plt.subplots(figsize=(18, 6))
ax.bar(df_sorted.AUX_Variable, df_sorted[plot_var], color="#4C72B0")
ax.set_ylabel("Test R²")
ax.set_ylim(-1, 1.0)
ax.set_title("Ridge Probe Performance per Auxiliary Variable")
plt.xticks(rotation=90)
ax.axhline(0.5, color="red", linestyle="--", label="R² = 0.5")
plt.tight_layout()
plt.show()

In [ ]:
[x for x in df_sorted.AUX_Variable if "test-aux" in x]

In [ ]:
df_sorted["is_test_aux"] = df_sorted.AUX_Variable.str.startswith("test-aux")
# df_sorted[:40]

In [ ]:
save_fig = True

if DATASET == "satbird-USA-summer":
    df_sorted["is_test_aux"] = False
    n_plots = 1
else:
    n_plots = 2

fig, ax = plt.subplots(1, n_plots, figsize=(4 * n_plots, 3))
if n_plots == 1:
    ax = [ax]  # make it iterable
plot_data = df_sorted[plot_var]
min_val = plot_data.min()
min_val = np.floor(min_val * 4) / 4  # round to nearest 0.25
bins = np.arange(min_val, 1.0 + 0.25, 0.25)  # bins from min_val to 1.0 in steps of 0.25

for i_plot, is_test_aux in enumerate([False, True]):
    if i_plot >= n_plots:
        break
    df_plot = df_sorted[df_sorted["is_test_aux"] == is_test_aux]
    ax[i_plot].hist(
        df_plot[plot_var],
        bins=bins,
        width=0.25,
        color="#81A0D2" if is_test_aux else "#B17171",
        edgecolor="black",
    )
    ax[i_plot].set_xlabel("R² linear probe")
    if i_plot == 0:
        ax[i_plot].set_ylabel("# of auxiliary variables")
    ax[i_plot].set_title(
        f"{'OOD' if is_test_aux else 'ID'} S2BMS auxiliary variable encoding\nin AlphaEarth embeddings",
        fontsize=10,
    )
    ax[i_plot].axvline(0.5, color="k", linestyle="--", linewidth=3, label="R² = 0.5")
    ax[i_plot].set_ylim(top=30)
    for sp in ["top", "right"]:
        ax[i_plot].spines[sp].set_visible(False)

if n_plots > 1:
    vu.add_panel_label(
        ax=ax[0],
        fig=fig,
        label_letter=None,
        label_ind=0,
        fontsize=14,
        x_offset=0.23,
        y_offset=0.07,
    )

    vu.add_panel_label(
        ax=ax[1],
        fig=fig,
        label_letter=None,
        label_ind=1,
        fontsize=14,
        x_offset=1.45,
        y_offset=0.07,
    )

if save_fig:
    name_fig = f"{DATASET}_linear_probe_aux_aef.pdf"
    for path_save in [f"/Users/tplas/repos/ms_aether_biodiv/figs/{name_fig}"]:
        plt.savefig(
            path_save,
            bbox_inches="tight",
        )

In [ ]:
## For all columns with R2 > 0.5, val_av=True, else False. Add that to JSON and write back.

import json

fp_cc = f"{DATA_DIR}/s2bms/concept_captions/v5.json"
assert os.path.exists(fp_cc), f"Concept captions file not found: {fp_cc}"
with open(fp_cc, "r") as f:
    concept_captions = json.load(f)

for i, dict_c in enumerate(concept_captions):
    aux_name = dict_c["col"]
    if aux_name in df_sorted["AUX_Variable"].values:
        r2_val = df_sorted.loc[df_sorted["AUX_Variable"] == aux_name, "Test_R2"].values[0]
        concept_captions[i]["val_av"] = bool(r2_val > 0.5)
    else:
        concept_captions[i]["val_av"] = False

new_fp_cc = f"{DATA_DIR}/s2bms/concept_captions/v6.json"
# with open(new_fp_cc, 'w') as f:
#     json.dump(concept_captions, f, indent=2)

In [ ]:
## Plot both test R2 and lc sum as a bar plot

# Butterfly barplot: Test R2 (up) vs lc_sum (down, log scale) per land-cover variable
lc_df = df_sorted[df_sorted["lc_sum"].notna()].sort_values("Test_R2", ascending=False)

fig, (ax_top, ax_bot) = plt.subplots(
    2,
    1,
    figsize=(14, 8),
    sharex=True,
    gridspec_kw={"height_ratios": [1, 1], "hspace": 0.05},
)

ax_top.bar(lc_df.AUX_Variable, lc_df.Test_R2, color="#4C72B0")
ax_top.set_ylabel("Test R²")
ax_top.axhline(0.5, color="black", linewidth=0.8)
ax_top.axhline(0, color="black", linewidth=0.8)

ax_bot.bar(lc_df.AUX_Variable, lc_df.lc_sum, color="#DD8452")
ax_bot.set_yscale("log")
ax_bot.invert_yaxis()
ax_bot.set_ylabel("LC sum (log)")
ax_bot.axhline(100, color="black", linewidth=0.8)

plt.xticks(rotation=90)
fig.suptitle("Ridge Probe Test R² vs. Land-Cover Sum (Train)")
plt.tight_layout()
plt.show()

In [ ]:
# df_sorted[:30]

import seaborn as sns

col_1, col_2 = "Test_R2", "rse"
inds_nonnan = df_sorted[col_1].notna() & df_sorted[col_2].notna()
r, p = pearsonr(df_sorted.loc[inds_nonnan, col_1], df_sorted.loc[inds_nonnan, col_2])
print(f"Pearson correlation between {col_1} and {col_2}: r={r:.4f}, p={p:.4e}")
df_sorted["lc_sum_bin"] = pd.cut(df_sorted["lc_sum"], bins=10, labels=False, include_lowest=True)
sns.scatterplot(
    data=df_sorted, x=col_1, y=col_2, hue="lc_sum_bin", palette="viridis", hue_norm=(0, 9)
)
# plt.ylim(-1.3, 1.0)

# plt.xlim(0, 500)

## RSE

In [ ]:
print("Shape of aux_train and aef_train:", aux_train.shape, aef_train.shape)

from scipy.spatial.distance import cdist, pdist

dist_aef = cdist(X_train, X_train, metric="euclidean")
upper_tri_indices = np.triu_indices(dist_aef.shape[0], k=1)
dist_aef = dist_aef[upper_tri_indices]

dict_rse = {}
for i_col, aux_name in tqdm(enumerate(kept_names)):
    dist_aux = cdist(
        y_train[:, i_col].reshape(-1, 1), y_train[:, i_col].reshape(-1, 1), metric="euclidean"
    )
    dist_aux = dist_aux[upper_tri_indices]
    corr, _ = pearsonr(dist_aef, dist_aux)
    dict_rse[aux_name] = corr
    # print(f"Pearson correlation between AEF and {kept_names[i_col]} distance matrices: {corr:.4f}")

    df_sorted["rse"] = df_sorted["AUX_Variable"].map(dict_rse)

# TMP Satbird

In [ ]:
fp_satbird = os.path.join(DATA_DIR, "satbird-USA-summer")
assert os.path.exists(fp_satbird)

data_satbird = pd.read_csv(os.path.join(fp_satbird, "model_ready_satbird-USA-summer.csv"))

In [ ]:
[x for x in data_satbird.columns if "aux" in x]